In [ ]:
# Import the libraries we need
import os
import re
import html
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
# Folders and settings (paths are relative to the notebooks folder)
DATA_DIR = "../data"
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
EMB_MODEL = "all-MiniLM-L6-v2"   # small sentence embedding model
SEED = 42

In [ ]:
# Clean a headline: fix html symbols and remove the news source at the end, like (Reuters)
def clean_text(text):
    text = re.sub(r"\s?#39;", "'", str(text)).replace("quot;", '"')
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)          # remove html tags
    text = text.replace("\\", " ")
    text = re.sub(r"\s*\([^)]*\)\s*$", "", text)  # remove (Reuters), (AFP), ...
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print(clean_text("Wall St. Bears Claw Back Into the Black (Reuters)"))

In [ ]:
# Load the AG News data. We use the headline (title) only, not the description
columns = ["label_id", "title", "description"]
train_df = pd.read_csv(f"{DATA_DIR}/train.csv", header=None, names=columns)
test_df = pd.read_csv(f"{DATA_DIR}/test.csv", header=None, names=columns)

for df in [train_df, test_df]:
    df["text"] = df["title"].apply(clean_text)
    df["label"] = df["label_id"].map(LABELS)

train_df = train_df[["text", "label"]]
test_df = test_df[["text", "label"]]

# Remove empty and duplicate headlines from the training data
train_df = train_df[train_df["text"].str.len() > 0].drop_duplicates()

print("train:", train_df.shape, " test:", test_df.shape)
train_df.head()

In [ ]:
# Split the training data: 90% train, 10% dev (dev is used to choose the best settings)
train, dev = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=SEED)
print("train:", len(train), " dev:", len(dev), " test:", len(test_df))
train["label"].value_counts()

In [ ]:
# Load the embedding model (it turns each headline into a vector of 384 numbers)
encoder = SentenceTransformer(EMB_MODEL)

# Encode the headlines. The vectors are saved in the data folder, so next time we just load them
def get_embeddings(df, name):
    path = f"{DATA_DIR}/emb_{name}.npy"
    if os.path.exists(path):
        emb = np.load(path)
        if len(emb) == len(df):
            return emb
    emb = encoder.encode(df["text"].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
    np.save(path, emb)
    return emb

X_train = get_embeddings(train, "train")
X_dev = get_embeddings(dev, "dev")
X_test = get_embeddings(test_df, "test")
print("train:", X_train.shape, " dev:", X_dev.shape, " test:", X_test.shape)

In [ ]:
# Training: a small neural network (MLP) on top of the embeddings
# It works better than Logistic Regression here because it can learn non-linear patterns
# early_stopping keeps 10% of the train data to stop training before overfitting
best_f1 = 0
best_model = None

for size in [256, 512, 1024]:
    model = MLPClassifier(hidden_layer_sizes=(size,), early_stopping=True, max_iter=50, random_state=SEED)
    model.fit(X_train, train["label"])

    pred = model.predict(X_dev)
    acc = accuracy_score(dev["label"], pred)
    f1 = f1_score(dev["label"], pred, average="macro")
    print(f"hidden size={size}  ->  acc={acc:.4f}  f1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_size = size

print("\nBest hidden size:", best_size, " dev F1:", round(best_f1, 4))

In [ ]:
# Save the best model in the models folder
joblib.dump(best_model, f"{MODEL_DIR}/emb_mlp.joblib")
print("model saved")

In [ ]:
# Testing: evaluate the best model on the test set (used only once, at the end)
y_test = test_df["label"]
y_pred = best_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 4))
print()
print(classification_report(y_test, y_pred, labels=LABEL_NAMES, digits=4))

In [ ]:
# Confusion matrix: rows are the true topic, columns are the predicted topic
cm = confusion_matrix(y_test, y_pred, labels=LABEL_NAMES)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title("Embeddings + MLP")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
# Error analysis: Business and World are the topics the model mixes up the most
results = test_df.copy()
results["pred"] = y_pred

print("Business -> World:", ((results["label"] == "Business") & (results["pred"] == "World")).sum())
print("World -> Business:", ((results["label"] == "World") & (results["pred"] == "Business")).sum())

mistakes = results[(results["label"] != results["pred"]) & results["label"].isin(["Business", "World"])]
mistakes.head(10)

In [ ]:
# Accuracy by headline length (number of words)
results["words"] = results["text"].str.split().str.len()
results["length"] = pd.cut(results["words"], bins=[0, 5, 8, 100], labels=["<=5", "6-8", ">8"])
results["correct"] = results["label"] == results["pred"]
results.groupby("length", observed=True)["correct"].agg(["count", "mean"]).round(4)

In [ ]:
# Try the model on new headlines
headlines = [
    "Oil prices jump after OPEC cuts production",
    "Real Madrid beat Barcelona 2-1 in El Clasico",
    "Apple unveils new chip for faster AI on phones",
]
emb = encoder.encode([clean_text(h) for h in headlines], normalize_embeddings=True)
for h, p in zip(headlines, best_model.predict(emb)):
    print(p, " <- ", h)